# Talking to an MCP Server: ChemE Tools
The Model Context Protocol (MCP) is an open protocol that lets an agent application discover and call tools exposed by a separate server process over a standard message format. In this demo, we launch a local Julia MCP server as a subprocess and drive a full session against it, watching every message cross the wire in both directions.

> __Learning Objectives__
>
> By the end of this demo, you will be able to:
> * __Launch and initialize a server:__ Start an MCP server as a subprocess over stdio and complete the initialization handshake that exchanges protocol version, capabilities, and server identity.
> * __Discover available tools:__ Use the `tools/list` request to read the name, description, and input schema of every tool the server exposes.
> * __Call tools and handle failures:__ Use `tools/call` to invoke a tool with arguments, and distinguish a protocol error from a tool execution error.

Let's get started!
___

## Setup, Data, and Prerequisites
This module ships a self-contained MCP implementation. The local Julia environment holds the client and server code in `src/`, and the `server.jl` entry point at the module root is the script the client launches as a subprocess. We load the environment with `Include.jl`, which activates the project, loads the required packages, and includes the `src/` files.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, and includes our code. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [1]:
include("Include.jl");

  Activating 

project at `~/Desktop/julia_work/CHEME-140-eCornell-Repository/courses/CHEME-142/module-4`


### Constants
Before we open a connection, we build the command that launches the server. The client spawns this command as a subprocess and speaks to it over the subprocess's standard input and output. The command names the Julia executable, activates this module's project, and runs `server.jl`.

> This launch command has the same shape as a command entry in a host application's MCP configuration file: an executable plus arguments that start a server process. A host reads such entries to know how to start each server it connects to.

In [2]:
server_command = `$(joinpath(Sys.BINDIR, "julia")) --project=$(_ROOT) $(_PATH_TO_SERVER)`;

## Task 1: Connect and Initialize
We open the connection by launching the server as a subprocess. The client and server communicate over stdio: the client writes JSON-RPC messages to the subprocess's standard input, and the server writes responses to its standard output, one message per line.

Every MCP session begins with an initialization handshake:

> __Initialize:__ The client sends an `initialize` request carrying its protocol version, capabilities, and client identity. The server replies with its own protocol version, capabilities, and a `serverInfo` block naming the server. The client then sends a `notifications/initialized` notification to confirm the handshake is complete; a notification carries no `id` and receives no response.

We pass `verbose = true` so the connection echoes every wire message: `→` marks a message the client sends, and `←` marks a message the server returns.

In [3]:
connection = connect(server_command, verbose = true);
response_initialize = initialize!(connection)

→ {"method":"initialize","id":1,"params":{"clientInfo":{"name":"cheme-142-notebook-client","version":"1.0.0"},"protocolVersion":"2025-06-18","capabilities":{}},"jsonrpc":"2.0"}

  Activating

 project at `~/Desktop/julia_work/CHEME-140-eCornell-Repository/courses/CHEME-142/module-4`


← {"id":1,"jsonrpc":"2.0","result":{"protocolVersion":"2025-06-18","capabilities":{"tools":{}},"serverInfo":{"name":"cheme-142-m4-cheme-tools","version":"1.0.0"}}}
→ {"method":"notifications/initialized","jsonrpc":"2.0"}


Dict{String, Any} with 3 entries:
  "id"      => 1
  "jsonrpc" => "2.0"
  "result"  => Dict{String, Any}("protocolVersion"=>"2025-06-18", "capabilities"=>Dict{String, Any}("tools"=>Dict{String, Any}()), "serverInfo"=>Dict{String, Any}("name"=>"cheme-142-m4-cheme-tools", …

The handshake response carries the server's identity in its `result`. Let's pull out the `serverInfo` block, which names the server and its version.

In [4]:
response_initialize["result"]["serverInfo"]

Dict{String, Any} with 2 entries:
  "name"    => "cheme-142-m4-cheme-tools"
  "version" => "1.0.0"

___

## Task 2: Discover the Tools
With the session initialized, the client asks the server what it can do. The `tools/list` request returns a descriptor for each tool: its `name`, a human-readable `description`, and an `inputSchema` written in JSON Schema. This is how an agent learns a server's capabilities at runtime, without documentation or a language-specific SDK.

In [5]:
response_tools = listtools(connection);
tools = response_tools["result"]["tools"];

→ {"method":"tools/list","id":2,"params":{},"jsonrpc":"2.0"}
← {"id":2,"jsonrpc":"2.0","result":{"tools":[{"name":"antoine_vapor_pressure","inputSchema":{"properties":{"T":{"type":"number","description":"Temperature in K, inside the species' valid range"},"species":{"type":"string","description":"Species name, e.g. water, acetone, ethanol, benzene"}},"required":["species","T"],"type":"object"},"description":"Saturation pressure (bar) of a named species at temperature T (K) from the Antoine equation."},{"name":"ideal_gas_solve","inputSchema":{"properties":{"T":{"type":"number","description":"Temperature in K"},"P":{"type":"number","description":"Pressure in Pa"},"V":{"type":"number","description":"Volume in m^3"},"n":{"type":"number","description":"Amount in mol"}},"required":[],"type":"object"},"description":"Solve the ideal gas law PV = nRT for the one variable not provided (SI units: Pa, m^3, mol, K)."},{"name":"molecular_weight","inputSchema":{"properties":{"formula":{"type":"string

In [6]:
let
    table = Matrix{String}(undef, length(tools), 2);
    for (i, tool) in enumerate(tools)
        table[i, 1] = tool["name"];
        table[i, 2] = tool["description"];
    end
    pretty_table(table; column_labels = ["name", "description"])
end

┌────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────┐
│                   name │                                                                                     description │
├────────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ antoine_vapor_pressure │    Saturation pressure (bar) of a named species at temperature T (K) from the Antoine equation. │
│        ideal_gas_solve │ Solve the ideal gas law PV = nRT for the one variable not provided (SI units: Pa, m^3, mol, K). │
│       molecular_weight │                      Compute the molar mass (g/mol) of a chemical formula, e.g. H2O or C6H12O6. │
└────────────────────────┴─────────────────────────────────────────────────────────────────────────────────────────────────┘


The table shows what each tool does, but not how to call it. The `inputSchema` is the contract an agent uses to construct arguments. Let's print one schema in full.

In [7]:
JSON.print(tools[1]["inputSchema"], 2) # antoine_vapor_pressure (tools are listed alphabetically)

{
  "properties": {
    "T": {
      "type": "number",
      "description": "Temperature in K, inside the species' valid range"
    },
    "species": {
      "type": "string",
      "description": "Species name, e.g. water, acetone, ethanol, benzene"
    }
  },
  "required": [
    "species",
    "T"
  ],
  "type": "object"
}


___

## Task 3: Call the Tools
To invoke a tool, the client sends a `tools/call` request naming the tool and passing an `arguments` object that conforms to the tool's input schema. The result carries a `content` array of typed items (text here) and an `isError` flag. We call `molecular_weight` with a chemical formula and parse the text payload it returns.

In [8]:
response_mw = calltool(connection, "molecular_weight", Dict("formula" => "C6H12O6"));
result_mw = JSON.parse(response_mw["result"]["content"][1]["text"])

→ {"method":"tools/call","id":3,"params":{"name":"molecular_weight","arguments":{"formula":"C6H12O6"}},"jsonrpc":"2.0"}

← {"id":3,"jsonrpc":"2.0","result":{"content":[{"text":"{\"units\":\"g/mol\",\"formula\":\"C6H12O6\",\"molar_mass\":180.156}","type":"text"}],"isError":false}}


Dict{String, Any} with 3 entries:
  "units"      => "g/mol"
  "formula"    => "C6H12O6"
  "molar_mass" => 180.156

The server computed a molar mass of 180.156 g/mol for glucose. Next we request a thermodynamic property that the server reads from a coefficient table it owns: the saturation pressure of acetone at 320 K from the Antoine equation.

In [9]:
response_psat = calltool(connection, "antoine_vapor_pressure", Dict("species" => "acetone", "T" => 320.0));
result_psat = JSON.parse(response_psat["result"]["content"][1]["text"])

→ {"method":"tools/call","id":4,"params":{"name":"antoine_vapor_pressure","arguments":{"T":320.0,"species":"acetone"}},"jsonrpc":"2.0"}
← {"id":4,"jsonrpc":"2.0","result":{"content":[{"text":"{\"units\":\"bar\",\"T\":320.0,\"Psat\":0.7261,\"species\":\"acetone\"}","type":"text"}],"isError":false}}


Dict{String, Any} with 4 entries:
  "units"   => "bar"
  "T"       => 320.0
  "Psat"    => 0.7261
  "species" => "acetone"

### When Calls Fail
Not every call succeeds, and MCP separates two kinds of failure:

> __Protocol error:__ The request itself was invalid, for example naming a tool the server does not have. The server returns a JSON-RPC error object with a numeric `code` (here `-32602`, invalid params) and no `result`.

> __Tool execution error:__ The request was valid and the tool ran, but the tool failed, for example an argument outside its valid range. The server returns a normal `result` with `isError` set to `true` and the failure text in `content`.

A host handles these differently: a protocol error signals a malformed request, while a tool execution error is a result the host reports back to the model. We trigger one of each below.

In [10]:
response_unknown = calltool(connection, "gibbs_energy", Dict("species" => "water"))

→ {"method":"tools/call","id":5,"params":{"name":"gibbs_energy","arguments":{"species":"water"}},"jsonrpc":"2.0"}


← {"error":{"message":"Unknown tool: gibbs_energy","code":-32602},"id":5,"jsonrpc":"2.0"}


Dict{String, Any} with 3 entries:
  "error"   => Dict{String, Any}("message"=>"Unknown tool: gibbs_energy", "code"=>-32602)
  "id"      => 5
  "jsonrpc" => "2.0"

In [11]:
response_range = calltool(connection, "antoine_vapor_pressure", Dict("species" => "water", "T" => 500.0))

→ {"method":"tools/call","id":6,"params":{"name":"antoine_vapor_pressure","arguments":{"T":500.0,"species":"water"}},"jsonrpc":"2.0"}
← {"id":6,"jsonrpc":"2.0","result":{"content":[{"text":"ArgumentError: T = 500.0 K is outside the valid range [255.9, 373.0] K for water","type":"text"}],"isError":true}}


Dict{String, Any} with 3 entries:
  "id"      => 6
  "jsonrpc" => "2.0"
  "result"  => Dict{String, Any}("content"=>Any[Dict{String, Any}("text"=>"ArgumentError: T = 500.0 K is outside the valid range [255.9, 373.0] K for water", "type"=>"text")], "isError"=>true)

Compare the two responses. `response_unknown` has an `error` field and no `result`: the server rejected the request because no such tool exists. `response_range` has a `result` with `isError` set to `true` and the failure message in its `content`: the tool was found and ran, but rejected the out-of-range temperature.

## Shut Down
The stdio transport has no dedicated shutdown message. Instead, the client closes the server's standard input, which signals end-of-file. The server's dispatch loop reads one line at a time and exits when its input reaches EOF, so closing stdin ends the session and lets the subprocess terminate. We close the connection and confirm the process has exited.

In [12]:
close(connection);
process_exited(connection.process)

true

## The Open-Source MCP Ecosystem
The server in this module is small enough to read end to end, but it speaks the same protocol as the wider MCP ecosystem. The reference servers, including Everything, Filesystem, Fetch, and Time, are published at [github.com/modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers) as readable examples. Official SDKs implement the protocol in several languages, and the specification is documented at [modelcontextprotocol.io](https://modelcontextprotocol.io). A client that drives our Julia server drives any of these the same way: initialize, list tools, call tools, shut down.
___

## Summary
This demo drove a full MCP session against a local server: launch and handshake, tool discovery, tool calls, both failure modes, and shutdown.

> __Key Takeaways:__
>
> * **Launch and handshake over stdio:** An MCP client starts the server as a subprocess and exchanges `initialize` and `notifications/initialized` messages over stdio before any tool call.
> * **Discover, then call:** The client reads each tool's JSON Schema from `tools/list` and uses it as the contract to construct arguments for a `tools/call` request.
> * **Two kinds of failure:** A protocol error returns a JSON-RPC error object with no `result`, while a tool execution error returns a normal `result` with `isError` set to `true`.

The same client operations you saw here carry over to the activities, where you drive the server yourself and then extend it with a new tool.
___